In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.3 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader, PdfWriter, PageObject, Transformation

def create_imposition_pdf(input_pdf_path, output_pdf_path):
    reader = PdfReader(input_pdf_path)
    writer = PdfWriter()

    total_pages = len(reader.pages)
    if total_pages != 40:
        raise ValueError("入力PDFは40ページである必要があります。")

    # A5の標準サイズ (ポイント単位: 420 x 595) -> A3は (1190 x 842)
    # ここでは入力ページのサイズを自動取得してA3キャンバスを作成します
    sample_page = reader.pages[0]
    w = float(sample_page.mediabox.width)
    h = float(sample_page.mediabox.height)

    # A3 1面のサイズ (横: 2w, 縦: 2h)
    a3_w, a3_h = w * 2, h * 2

    # ページ配置のインデックス配列作成 (0-indexed)
    sheets = total_pages // 8

    for i in range(sheets):
        # 表面の配置 [上左, 上右, 下左, 下右] (1-indexedを0-indexedに変換)
        front_indices = [
            (40 - 2*i) - 1,   (2*i + 1) - 1,
            (30 - 2*i) - 1,   (11 + 2*i) - 1
        ]
        # 裏面の配置 [上左, 上右, 下左, 下右]
        back_indices = [
            (2*i + 2) - 1,    (40 - 2*i - 1) - 1,
            (12 + 2*i) - 1,   (29 - 2*i) - 1
        ]

        for indices in [front_indices, back_indices]:
            # 新しいA3キャンバスを作成
            new_page = PageObject.create_blank_page(width=a3_w, height=a3_h)

            # 4つの位置に配置 (上左, 上右, 下左, 下右)
            positions = [
                (0, h),     # 上左
                (w, h),     # 上右
                (0, 0),     # 下左
                (w, 0)      # 下右
            ]

            for idx, (x, y) in zip(indices, positions):
                p = reader.pages[idx]
                # 位置を平行移動して合成
                new_page.merge_transformed_page(
                    p,
                    Transformation().translate(tx=x, ty=y)
                )

            writer.add_page(new_page)

    with open(output_pdf_path, "wb") as f:
        writer.write(f)

# 使用例
create_imposition_pdf("2026.pdf", "output_a3_imposed.pdf")